# EDA - Squad 08

### Leitura das bases

In [ ]:
# Criar dicionário para armazenar DataFrames
dataframes = {}
caminho_root = '/Volumes/hackathon_2025/default/source/'

# base_dados_cadastrais
try:
    df = spark.read.parquet(f'{caminho_root}base_dados_cadastrais/')
    dataframes['base_dados_cadastrais'] = df
    print(f'base_dados_cadastrais:{df.count():,} linhas, {len(df.columns)} colunas')
except Exception as e:
    print(f'Erro: {str(e)[:100]}')

# base_score_bureau_movel
try:
    df = spark.read.parquet(f'{caminho_root}base_score_bureau_movel/')
    dataframes['base_score_bureau_movel'] = df
    print(f'base_score_bureau_movel:{df.count():,} linhas, {len(df.columns)} colunas')
except Exception as e:
    print(f'Erro: {str(e)[:100]}')

# base_score_bureau_movel_full (NOVA)
try:
    df = spark.read.parquet(f'{caminho_root}base_score_bureau_movel_full/')
    dataframes['base_score_bureau_movel_full'] = df
    print(f'base_score_bureau_movel_full:{df.count():,} linhas, {len(df.columns)} colunas')
except Exception as e:
    print(f'Erro: {str(e)[:100]}')

# base_telco
try:
    df = spark.read.parquet(f'{caminho_root}base_telco/')
    dataframes['base_telco'] = df
    print(f'base_telco:{df.count():,} linhas, {len(df.columns)} colunas')
except Exception as e:
    print(f'Erro: {str(e)[:100]}')

# csv de bases_recarga
csv_files = [
    'BI_DIM_CANAL_AQUISICAO_CREDITO',
    'BI_DIM_FORMA_PAGAMENTO',
    'BI_DIM_INSTITUICAO',
    'BI_DIM_PLANO_PRECO',
    'BI_DIM_PLATAFORMA',
    'BI_DIM_PROMOCAO_CREDITO',
    'BI_DIM_STATUS_PLATAFORMA',
    'BI_DIM_TECNOLOGIA',
    'BI_DIM_TIPO_CREDITO',
    'BI_DIM_TIPO_INSERCAO',
    'BI_DIM_TIPO_RECARGA'
]

for csv_file in csv_files:
    try:
        df = spark.read.csv(f'{caminho_root}bases_recarga/{csv_file}.csv', header=True, inferSchema=True)
        dataframes[csv_file] = df
        print(f'{csv_file}: {df.count():,} linhas')
    except Exception as e:
        print(f'{csv_file}: {str(e)[:50]}')

# BI_DIM_TIPO_FATURAMENTO
try:
    df = spark.read.csv(f'{caminho_root}book_atraso/BI_DIM_TIPO_FATURAMENTO.csv', header=True, inferSchema=True)
    dataframes['BI_DIM_TIPO_FATURAMENTO'] = df
    print(f'BI_DIM_TIPO_FATURAMENTO: {df.count():,} linhas')
except Exception as e:
    print(f'Erro: {str(e)[:100]}')

# parquet - book_atraso/dados_faturamento/
try:
    df = spark.read.parquet(f'{caminho_root}book_atraso/dados_faturamento/')
    dataframes['dados_faturamento'] = df
    print(f'dados_faturamento: {df.count():,} linhas, {len(df.columns)} colunas')
except Exception as e:
    print(f'Erro: {str(e)[:100]}')

# parquet - bases_recarga/BI_FP_ASS_RECARGA_CMV_NOVA/
try:
    df = spark.read.parquet(f'{caminho_root}bases_recarga/BI_FP_ASS_RECARGA_CMV_NOVA/')
    dataframes['BI_FP_ASS_RECARGA_CMV_NOVA'] = df
    print(f'BI_FP_ASS_RECARGA_CMV_NOVA: {df.count():,} linhas, {len(df.columns)} colunas')
except Exception as e:
    print(f'Erro: {str(e)[:100]}')

# parquet - book_pagamento/dados_pagamento/
import os
caminho_pagamento = f'{caminho_root}book_pagamento/dados_pagamento/'

try:
    # Tentar carregar com wildcard
    df = spark.read.parquet(f'{caminho_pagamento}*.parquet')
    dataframes['dados_pagamento'] = df
    print(f'dados_pagamento (wildcard): {df.count():,} linhas, {len(df.columns)} colunas')
except Exception as e:
    print(f'Wildcard falhou, tentando listar arquivos...')

    # Se wildcard falhar, tentar carregar cada arquivo
    try:
        arquivos = os.listdir(caminho_pagamento)
        print(f'Encontrados {len(arquivos)} arquivos:')

        for arquivo in arquivos:
            if arquivo.endswith('.parquet'):
                print(f"     → {arquivo}")
                try:
                    caminho_completo = os.path.join(caminho_pagamento, arquivo)
                    df = spark.read.parquet(caminho_completo)

                    # Usar nome do arquivo como chave
                    nome_tabela = arquivo.replace('.parquet', '').replace('-c000', '')
                    if nome_tabela not in dataframes:  # Evitar duplicatas
                        dataframes[nome_tabela] = df

                    print(f'{nome_tabela}: {df.count():,} linhas')
                except Exception as e2:
                    print(f'Erro: {str(e2)[:50]}')
    except Exception as e:
        print(f'Erro ao listar diretório: {str(e)[:100]}')

# Converter todos os nomes dos dataframes para minúsculas
dataframes = {chave.lower(): df for chave, df in dataframes.items()}

### Metadados

In [ ]:
# importar biblioteca
from pyspark.sql.functions import col, count, when, lit


# Lista para armazenar metadados
metadata = []

# Iterar sobre cada dataframe
for nome_df, df in dataframes.items():
    total_linhas = df.count()

    # Analisar cada coluna
    for coluna in df.columns:
        tipo = df.schema[coluna].dataType

        # Contar nulos
        qt_nulos = df.filter(col(coluna).isNull()).count()
        percent_nulos = (qt_nulos / total_linhas) * 100 if total_linhas > 0 else 0

        # Cardinalidade (valores únicos)
        cardinalidade = df.select(coluna).distinct().count()

        # Adicionar à lista
        metadata.append({
            'nome_dataframe': nome_df,
            'nome_variavel': coluna,
            'tipo': str(tipo),
            'qt_nulos': int(qt_nulos),
            'percent_nulos': round(percent_nulos, 2),
            'cardinalidade': int(cardinalidade)
        })

# Converter para Spark DataFrame
df_resultado = spark.createDataFrame(metadata)

# Exibir
display(df_resultado)

### Tratamento dos dados

In [ ]:
from pyspark.sql.functions import to_date, col, substring, concat, lit

total_dat = 0
total_datadenascimento = 0
total_val = 0
total_safra = 0
total_int = 0

# Lista de colunas para converter em integer
colunas_integer = [
    'FLAG_INSTALACAO', 'FPD', 'SCORE_01', 'SCORE_02', 'FLAG_SOS'
]

# Nome da coluna de safra
COLUNA_SAFRA = 'SAFRA'

for nome_df, df in dataframes.items():
    # 1. DAT_ para date
    colunas_data = [c for c in df.columns if c.startswith('DAT_')]
    for col_data in colunas_data:
        try:
            df = df.withColumn(col_data, to_date(col(col_data), 'dd/MM/yyyy'))
            total_dat += 1
        except:
            try:
                df = df.withColumn(col_data, to_date(col(col_data), 'yyyy-MM-dd'))
                total_dat += 1
            except:
                pass
    
    # 2. DATADENASCIMENTO para date
    if 'DATADENASCIMENTO' in df.columns:
        try:
            df = df.withColumn('DATADENASCIMENTO', to_date(col('DATADENASCIMENTO'), 'dd/MM/yyyy'))
            total_datadenascimento += 1
        except:
            try:
                df = df.withColumn('DATADENASCIMENTO', to_date(col('DATADENASCIMENTO'), 'yyyy-MM-dd'))
                total_datadenascimento += 1
            except:
                pass
    
    # 3. VAL_ para float
    colunas_valor = [c for c in df.columns if c.startswith('VAL_')]
    for col_valor in colunas_valor:
        try:
            df = df.withColumn(col_valor, col(col_valor).cast('float'))
            total_val += 1
        except:
            pass
    
    # 4. SAFRA (YYYYMM paraYYYY/MM)
    if COLUNA_SAFRA in df.columns:
        try:
            df = df.withColumn(
                COLUNA_SAFRA,
                concat(
                    substring(col(COLUNA_SAFRA), 1, 4),
                    lit('/'),
                    substring(col(COLUNA_SAFRA), 5, 2)
                )
            )
            total_safra += 1
        except:
            pass
    
    # 5. Colunas para integer
    for col_int in colunas_integer:
        if col_int in df.columns:
            try:
                df = df.withColumn(col_int, col(col_int).cast('integer'))
                total_int += 1
            except:
                pass
    
    dataframes[nome_df] = df

# Print final
print(f"DAT_ para date: {total_dat}")
print(f"DATADENASCIMENTO para date: {total_datadenascimento}")
print(f"VAL_ para float: {total_val}")
print(f"SAFRA para YYYY/MM: {total_safra}")
print(f"Colunas explícitas para integer: {total_int}")
print(f"Colunas nº inteiro: {colunas_integer}")
print(f"TOTAL CONVERTIDAS: {total_dat + total_datadenascimento + total_val + total_safra + total_int}")

### Análise univarida de variáveis numéricas

Estatísticas descritivas para colunas numéricas

In [ ]:
from pyspark.sql.functions import countDistinct
import pandas as pd


for nome_df, df in dataframes.items():
    # COLUNAS NUMÉRICAS 
    colunas_numericas = [f.name for f in df.schema.fields 
                         if 'Integer' in str(f.dataType) or 'Double' in str(f.dataType) or 'Long' in str(f.dataType or 'Float' in str(f.dataType))]
    
    if colunas_numericas:
        print(f"{nome_df.upper()}")
        
        # Descrever (count, mean, stddev, min, 25%, 50%, 75%, max)
        df_desc = df.select(colunas_numericas).describe()
        
        # Converter para pandas
        df_desc_pd = df_desc.toPandas().set_index('summary').T
        
        # Adicionar colunas extras (unique, top, freq)
        for col_name in colunas_numericas:
            unique = df.select(countDistinct(col_name)).collect()[0][0]
            top_freq = df.groupBy(col_name).count().orderBy('count', ascending=False).first()
            
            df_desc_pd.loc[col_name, 'unique'] = unique
            df_desc_pd.loc[col_name, 'top'] = top_freq[0]
            df_desc_pd.loc[col_name, 'freq'] = top_freq[1]
        
        # Reordenar colunas
        ordem = ['count', 'unique', 'top', 'freq', 'mean', 'stddev', 'min', '25%', '50%', '75%', 'max']
        ordem_final = [c for c in ordem if c in df_desc_pd.columns]
        
        display(df_desc_pd[ordem_final])
    else:
        print(f"\n{nome_df}: Nenhuma coluna numérica encontrada")

In [ ]:
Boxplot

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql.types import IntegerType, LongType, DoubleType, FloatType

def boxplots_var_num_spark(df_spark, nome_df="", sample_size=10000):
    """
    Plota boxplots para colunas numéricas de um Spark DataFrame.
    
    :param df_spark: Spark DataFrame
    :param nome_df: Nome do dataframe
    :param sample_size: Tamanho da amostra (para dataframes muito grandes)
    """
    
    # IDENTIFICAR COLUNAS NUMÉRICAS
    colunas_numericas = [f.name for f in df_spark.schema.fields 
                         if isinstance(f.dataType, (IntegerType, LongType, DoubleType, FloatType))]
    
    if not colunas_numericas:
        print(f"Nenhuma coluna numérica em {nome_df}")
        return
    
    # CONVERTER PARA PANDAS (com sampling se necessário)
    total_linhas = df_spark.count()
    
    if total_linhas > sample_size:
        print(f"{nome_df}: {total_linhas:,} linhas. Usando amostra de {sample_size:,} para melhor performance")
        df_pandas = df_spark.select(colunas_numericas).sample(fraction=sample_size/total_linhas).toPandas()
    else:
        df_pandas = df_spark.select(colunas_numericas).toPandas()
    
    # ===== CRIAR PAINEL =====
    nrows = len(colunas_numericas) // 3 + (len(colunas_numericas) % 3 > 0)
    fig, axes = plt.subplots(nrows=nrows, ncols=3, figsize=(14, nrows * 4))
    
    if nrows == 1:
        axes = axes.reshape(1, -1)
    
    plt.tight_layout(pad=4)
    sns.set_style("whitegrid")
    
    # ===== PLOTAR =====
    for i, column in enumerate(colunas_numericas):
        row = i // 3
        col = i % 3
        
        sns.boxplot(data=df_pandas[column], ax=axes[row, col], color="skyblue")
        axes[row, col].set_title(f'{column}', fontdict={'fontsize': 14, 'fontweight': 'bold'})
        axes[row, col].set_ylabel('')
    
    # ===== REMOVER VAZIOS =====
    for j in range(len(colunas_numericas), nrows * 3):
        row = j // 3
        col = j % 3
        fig.delaxes(axes[row, col])
    
    fig.suptitle(f"Análise descritiva - BoxPlot - {nome_df}", fontsize=20, fontweight='bold', y=0.05)
    plt.show()

# EXECUTAR PARA TODOS DATAFRAMES
for nome_df, df in dataframes.items():
    boxplots_var_num_spark(df, nome_df)

Histograma

In [ ]:
def histogramas_var_num_spark(df_spark, nome_df="", sample_size=50000, bins=30):
    """
    Plota histogramas com KDE para colunas numéricas de um Spark DataFrame.
    
    :param df_spark: Spark DataFrame
    :param nome_df: Nome do dataframe
    :param sample_size: Tamanho da amostra (para dataframes muito grandes)
    :param bins: Número de bins do histograma
    """
    
    # IDENTIFICAR COLUNAS NUMÉRICAS
    colunas_numericas = [f.name for f in df_spark.schema.fields 
                         if isinstance(f.dataType, (IntegerType, LongType, DoubleType, FloatType))]
    
    if not colunas_numericas:
        print(f"{nome_df}: Nenhuma coluna numérica")
        return
    
    # CONVERTER PARA PANDAS (com sampling se necessário)
    total_linhas = df_spark.count()
    
    if total_linhas > sample_size:
        print(f"{nome_df}: {total_linhas:,} linhas. Usando amostra de {sample_size:,}")
        df_pandas = df_spark.select(colunas_numericas).sample(fraction=sample_size/total_linhas).toPandas()
    else:
        df_pandas = df_spark.select(colunas_numericas).toPandas()
        print(f"{nome_df}: {total_linhas:,} linhas")
    
    #  CRIAR PAINEL 
    nrows = len(colunas_numericas) // 3 + (len(colunas_numericas) % 3 > 0)
    fig, axes = plt.subplots(nrows=nrows, ncols=3, figsize=(14, nrows * 4))
    
    if nrows == 1:
        axes = axes.reshape(1, -1)
    
    plt.tight_layout(pad=4)
    sns.set_style("whitegrid")
    
    # PLOTAR HISTOGRAMAS COM KDE 
    for i, column in enumerate(colunas_numericas):
        row = i // 3
        col = i % 3
        
        try:
            sns.histplot(data=df_pandas[column], ax=axes[row, col], 
                        color="skyblue", bins=bins, kde=True)
            axes[row, col].set_title(f'{column}', fontdict={'fontsize': 14, 'fontweight': 'bold'})
            axes[row, col].set_ylabel('Frequência')
            axes[row, col].tick_params(axis='both', which='major', labelsize=12)
        except Exception as e:
            axes[row, col].text(0.5, 0.5, f'Erro ao plotar\n{str(e)[:30]}', 
                               ha='center', va='center', transform=axes[row, col].transAxes)
            axes[row, col].set_title(f'{column} (erro)', fontdict={'fontsize': 12, 'fontweight': 'bold'})
    
    #  REMOVER VAZIOS 
    for j in range(len(colunas_numericas), nrows * 3):
        fig.delaxes(axes.flatten()[j])
    
    fig.suptitle(f"Análise descritiva - Histograma com KDE - {nome_df}", 
                fontsize=18, fontweight='bold', y=1.05)
    
    plt.show()

#  EXECUTAR PARA TODOS 
for nome_df, df in dataframes.items():
    histogramas_var_num_spark(df, nome_df, sample_size=50000, bins=30)